In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel
import numpy as np
import gc
import warnings
import random
from tqdm import tqdm
warnings.filterwarnings('ignore')


def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_and_prepare_reasonable_dataset(file_path, test_size=0.2, min_samples_per_author=50, max_authors=100):
    """
    Load Google Jam dataset with reasonable parameters
    """
    print("Loading dataset...")
    data = pd.read_csv(file_path)
    
    data = data.dropna(subset=['flines', 'username'])
    data['flines'] = data['flines'].astype(str)
    data = data[data['flines'].str.strip() != '']
    
    author_counts = data['username'].value_counts()
    
    valid_authors = author_counts[author_counts >= min_samples_per_author].index
    filtered_data = data[data['username'].isin(valid_authors)]
    
    if len(valid_authors) > max_authors:
        top_authors = author_counts.head(max_authors).index
        filtered_data = filtered_data[filtered_data['username'].isin(top_authors)]
    
    label_encoder = LabelEncoder()
    filtered_data['EncodedLabels'] = label_encoder.fit_transform(filtered_data['username'])
    num_classes = len(label_encoder.classes_)
    
    train_data, test_data = custom_stratified_split(filtered_data, test_size=test_size)
    
    return train_data, test_data, label_encoder, num_classes

def custom_stratified_split(data, test_size=0.2):
    """Custom stratified split that ensures proper distribution"""
    train_data = []
    test_data = []
    
    grouped = data.groupby('username')
    
    for author, group in grouped:
        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        
        n_test = max(1, int(len(group) * test_size))
        
        if len(group) - n_test < 2:
            n_test = max(1, len(group) - 2)
        
        test_samples = group.iloc[:n_test]
        train_samples = group.iloc[n_test:]
        
        test_data.append(test_samples)
        train_data.append(train_samples)
    
    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)
    
    train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
    test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return train_data, test_data


class AuthorshipDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = dataframe["flines"].tolist()
        self.labels = dataframe["EncodedLabels"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        tokens = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


class DeepSeekCoderAuthorClassifier(nn.Module):
    def __init__(self, num_classes, model_name="deepseek-ai/deepseek-coder-1.3b-base"):
        super(DeepSeekCoderAuthorClassifier, self).__init__()
        
        print(f"Loading {model_name}...")
        self.encoder = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.encoder.config.hidden_size
        
        self.dropout = nn.Dropout(0.1)
        
        self.classifier = nn.Linear(hidden_size, num_classes)
        
        print(f"Model loaded. Hidden size: {hidden_size}, Num classes: {num_classes}")
        
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
        print(f"Total parameters: {total_params:,}")
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        last_hidden_state = outputs.last_hidden_state
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        
        sum_embeddings = torch.sum(last_hidden_state * attention_mask_expanded, dim=1)
        
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        
        mean_pooled = sum_embeddings / sum_mask
        
        pooled_output = self.dropout(mean_pooled)
        logits = self.classifier(pooled_output)
        
        return logits


def train_deepseekcoder_model(train_data, test_data, num_classes, seed=42):
    """Train DeepSeekCoder model for authorship attribution"""
    
    print(f"\n{'='*60}")
    print(f"Training DeepSeekCoder with seed: {seed}")
    print(f"{'='*60}")
    
    set_seed(seed)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    print("\nInitializing DeepSeekCoder model...")
    
    model_name = "deepseek-ai/deepseek-coder-1.3b-base"
    
    try:
        model = DeepSeekCoderAuthorClassifier(num_classes, model_name).to(device)
    except:
        print("Memory error with 1.3B model, trying smaller model...")
        model_name = "deepseek-ai/deepseek-coder-1b-base"
        model = DeepSeekCoderAuthorClassifier(num_classes, model_name).to(device)
    
    print(f"\nLoading tokenizer for {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("\nCreating datasets...")
    train_dataset = AuthorshipDataset(train_data, tokenizer, max_length=256)
    test_dataset = AuthorshipDataset(test_data, tokenizer, max_length=256)
    
    batch_size = 2  
    gradient_accumulation_steps = 4  
    effective_batch_size = batch_size * gradient_accumulation_steps
    
    print(f"Batch size: {batch_size}")
    print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
    print(f"Effective batch size: {effective_batch_size}")
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True if torch.cuda.is_available() else False,
        num_workers=0  
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=2,  
        shuffle=False,
        pin_memory=True if torch.cuda.is_available() else False,
        num_workers=0
    )
    
    print("\nSetting up training...")
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-5,  
        weight_decay=0.01
    )
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=3,  
        eta_min=1e-6
    )
    
    criterion = nn.CrossEntropyLoss()
    
    print("\nStarting training...")
    epochs = 3  
    model.train()
    global_step = 0
    train_losses = []
    
    for epoch in range(epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch + 1}/{epochs}")
        print(f"{'='*50}")
        
        model.train()
        epoch_loss = 0
        optimizer.zero_grad()
        
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training", leave=False), 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss = loss / gradient_accumulation_steps 
            
            loss.backward()
            
            if step % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
            
            epoch_loss += loss.item() * gradient_accumulation_steps
            
            if step % 20 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        avg_epoch_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_epoch_loss)
        print(f"\nEpoch {epoch+1} completed. Average Loss: {avg_epoch_loss:.4f}")
    
    print("\nSaving final model...")
    model_path = f"/home/aman_swaraj/Downloads/Codelite/deepseek_coder_authorship_seed{seed}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")
    
    print("\nStarting evaluation...")
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.extend(probs.cpu().numpy())
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f"\n{'='*50}")
    print(f"Evaluation Results")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Total test samples: {len(all_labels)}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    return accuracy, all_probs, all_preds, all_labels


def run_deepseekcoder_multi_seed_experiment(num_seeds=10):
    """Run multi-seed experiment with DeepSeekCoder"""
    
    print("\n" + "="*80)
    print("DEEPSEEK-CODER AUTHORSHIP ATTRIBUTION EXPERIMENT")
    print("="*80)
    
    SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555][:num_seeds]
    
    file_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv"
    print("\nLoading dataset...")
    
    train_data, test_data, label_encoder, num_classes = load_and_prepare_reasonable_dataset(
        file_path, 
        test_size=0.2,
        min_samples_per_author=2,
        max_authors=1000  
    )
    
    print(f"\nDataset Statistics:")
    print(f"  Number of authors: {num_classes}")
    print(f"  Training samples: {len(train_data)}")
    print(f"  Test samples: {len(test_data)}")
    print(f"  Number of seeds: {num_seeds}")
    
    all_results = []
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'='*60}")
        print(f"SEED {seed_idx+1}/{num_seeds}: {seed}")
        print(f"{'='*60}")
        
        try:
            accuracy, probs, preds, true_labels = train_deepseekcoder_model(
                train_data, test_data, num_classes, seed
            )
            
            seed_results = {
                'seed': seed,
                'accuracy': accuracy,
                'num_authors': num_classes,
                'train_samples': len(train_data),
                'test_samples': len(test_data)
            }
            
            all_results.append(seed_results)
            
            print(f"\nDeepSeekCoder Accuracy: {accuracy:.4f}")
            
        except Exception as e:
            print(f"Error with seed {seed}: {e}")
            continue
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*80)
    
    if not all_results:
        print("No successful runs!")
        return None
    
    results_df = pd.DataFrame(all_results)
    
    print("\nAverage Performance Across Seeds:")
    print("-" * 60)
    
    summary_stats = {
        'Metric': ['Accuracy', 'Number of Authors', 'Training Samples', 'Test Samples'],
        'Mean': [
            results_df['accuracy'].mean(),
            results_df['num_authors'].mean(),
            results_df['train_samples'].mean(),
            results_df['test_samples'].mean()
        ],
        'Std': [
            results_df['accuracy'].std(),
            0,  
            0,  
            0   
        ],
        'Min': [
            results_df['accuracy'].min(),
            results_df['num_authors'].min(),
            results_df['train_samples'].min(),
            results_df['test_samples'].min()
        ],
        'Max': [
            results_df['accuracy'].max(),
            results_df['num_authors'].max(),
            results_df['train_samples'].max(),
            results_df['test_samples'].max()
        ]
    }
    
    summary_df = pd.DataFrame(summary_stats)
    print(summary_df.to_string(index=False))
    
    print("\n" + "="*60)
    print("SAVING RESULTS")
    print("="*60)
    
    results_path = "/home/aman_swaraj/Downloads/Codelite/deepseekcoder_authorship_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Detailed results saved to: {results_path}")
    
    summary_path = "/home/aman_swaraj/Downloads/Codelite/deepseekcoder_authorship_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary statistics saved to: {summary_path}")
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        print("\nGenerating visualizations...")
        
        plt.figure(figsize=(10, 6))
        
        seeds_list = [f"Seed {s}" for s in results_df['seed']]
        x = np.arange(len(seeds_list))
        
        bars = plt.bar(x, results_df['accuracy'], color='#4ECDC4', alpha=0.8)
        
        plt.xlabel('Random Seed')
        plt.ylabel('Accuracy')
        plt.title(f'DeepSeekCoder Authorship Attribution Across {len(seeds_list)} Seeds')
        plt.xticks(x, seeds_list)
        plt.grid(True, alpha=0.3, linestyle='--')
        
        for i, (bar, acc) in enumerate(zip(bars, results_df['accuracy'])):
            plt.text(bar.get_x() + bar.get_width()/2, 
                    bar.get_height() + 0.005,
                    f'{acc:.4f}', ha='center', va='bottom', fontsize=9)
        
        avg_acc = results_df['accuracy'].mean()
        plt.axhline(y=avg_acc, color='red', linestyle='--', linewidth=2, 
                   label=f'Average: {avg_acc:.4f}')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('deepseekcoder_authorship_performance.png', dpi=300, bbox_inches='tight')
        print("  Saved: deepseekcoder_authorship_performance.png")
        
        plt.show()
        
    except ImportError:
        print("\nVisualization libraries not available. Skipping plots.")



if __name__ == "__main__":
    print("Authorship Attribution with DeepSeekCoder")
    print("="*80)
    
    num_seeds = 10  
    results_df, summary_df = run_deepseekcoder_multi_seed_experiment(num_seeds=num_seeds)
    
    if results_df is not None:
        print("\n" + "="*80)
        print("FINAL SUMMARY")
        print("="*80)
        print(f"Dataset: Google Code Jam")
        print(f"Number of seeds evaluated: {len(results_df)}")
        print(f"Average DeepSeekCoder Accuracy: {results_df['accuracy'].mean():.4f}")
        print(f"Number of authors: {int(results_df['num_authors'].mean())}")